# Relevance filter
Leakage-free, checkpointed rebuild (protocol in `notebooks/AGENTS.md`, values in `notebooks/filter.toml`, logic in `notebooks/filtering.py`): 7 embedders x {LR, GNB, RF, XGB} (StandardScaler + classifier) scored with grouped 5-fold CV on the train split, Bayesian tuning of the selected candidate, refit, one test report, Cotov external evaluation with the deployed filter. Reads only the shared split and embedding artifacts.

In [ ]:
import json
import os
import platform
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
os.chdir(os.getenv("ROOT"))

from device import select_device
from notebooks import checkpoints as ck
from notebooks import embeddings, filtering as fl, splits, taxonomy
from notebooks.textprep import TEXTPREP_VERSION

In [ ]:
def env_flag(name):
    return os.getenv(name, "").strip().lower() in ("1", "true", "yes")


SMOKE = env_flag("FILTER_SMOKE")  # tiny NON-PRODUCTION check; never writes the production outputs
OVERWRITE = env_flag("OVERWRITE")
ALLOW_UNRESOLVED = env_flag("ALLOW_UNRESOLVED_EMBEDDERS")
ALLOW_NO_COTOV = env_flag("FILTER_ALLOW_NO_COTOV")  # production run may finish without the Cotov evaluation (recorded as blocked)
RANDOM_STATE = int(os.environ["RANDOM_STATE"])
FROZEN_SEED = 42
SPLIT_MANIFEST_CSV = Path(os.environ["SPLIT_MANIFEST_CSV"])
SPLIT_TRAIN_CSV = Path(os.environ["SPLIT_TRAIN_CSV"])
SPLIT_TEST_CSV = Path(os.environ["SPLIT_TEST_CSV"])
SPLIT_MANIFEST_SHA256 = os.environ["SPLIT_MANIFEST_SHA256"]
EMBEDDINGS_DIR = Path(os.environ["EMBEDDINGS_DIR"])
COTOV_CSV = os.getenv("COTOV_CSV")  # labels (Related) only; the features come from the cotov artifact
PRODUCTION_PICKLE = os.environ["OPTIMAL_FILTER_PICKLE"]
PRODUCTION_CSV = os.environ["FILTERS_PERFORMANCES_CSV"]
if SMOKE:
    SMOKE_DIR = Path(os.environ["FILTER_SMOKE_OUT_DIR"])
    SMOKE_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PICKLE, OUT_CSV = str(SMOKE_DIR / "smoke_filter.pkl"), str(SMOKE_DIR / "smoke_filters_performances.csv")
    fl.refuse_production_output(OUT_PICKLE, PRODUCTION_PICKLE)
    fl.refuse_production_output(OUT_CSV, PRODUCTION_CSV)
else:
    OUT_PICKLE, OUT_CSV = PRODUCTION_PICKLE, PRODUCTION_CSV
if not OVERWRITE and (Path(OUT_PICKLE).exists() or Path(OUT_CSV).exists()):
    raise FileExistsError(f"{OUT_PICKLE} / {OUT_CSV} exist; set OVERWRITE=true to replace them")

CFG = fl.effective_config(fl.load_config("notebooks/filter.toml"), SMOKE)
CANDIDATES = fl.build_candidates(CFG)
EMBEDDER_NAMES = CFG["selection"]["embedders"]
DEVICE = select_device()
MAX_EVALS = int(CFG["tuning"]["max_evals"])
BOOT = CFG["bootstrap"]  # B, seed, level of the 95% bootstrap CIs
print(f"{'NON-PRODUCTION SMOKE' if SMOKE else 'PRODUCTION'}: {len(CANDIDATES)} candidates, max_evals {MAX_EVALS}, device {DEVICE}")

In [ ]:
# Data: split files verified against the pinned manifest; labels from taxonomy.as_bool; folds = manifest inner_fold.
train_df = pd.read_csv(SPLIT_TRAIN_CSV)
test_df = pd.read_csv(SPLIT_TEST_CSV)
manifest = splits.read_manifest(SPLIT_MANIFEST_CSV)
full = pd.concat([train_df, test_df], ignore_index=True)
splits.verify_manifest(full, manifest, expected_seed=FROZEN_SEED, expected_manifest_sha256=SPLIT_MANIFEST_SHA256)
COLS = manifest.attrs["metadata"]["columns"]
S = fl.prepare_split(train_df, test_df, manifest, COLS["id"], COLS["relevant"], taxonomy.as_bool)
CORPUS_IDS = full[COLS["id"]].astype(str).tolist()
assert len(S.train_ids) + len(S.test_ids) == len(CORPUS_IDS)
assert len(set(S.folds.tolist())) == CFG["selection"]["cv_folds"], "inner folds != cv_folds"
print(f"manifest verified ({SPLIT_MANIFEST_SHA256[:12]}...): train {len(S.train_ids)} (relevant {S.y_train.mean():.3f}), "
      f"test {len(S.test_ids)} (relevant {S.y_test.mean():.3f}), folds {sorted(set(S.folds.tolist()))}")

In [ ]:
# Embedding artifacts (production only; an 'unresolved' fingerprint fails closed unless ALLOW_UNRESOLVED_EMBEDDERS=1).
ARTS = {}
for name in EMBEDDER_NAMES:
    ARTS[name] = embeddings.load_artifact(EMBEDDINGS_DIR, embeddings.artifact_name("corpus", name),
                                          expected_row_ids=CORPUS_IDS, allow_unresolved=ALLOW_UNRESOLVED)
X_TRAIN = {n: a.select(S.train_ids) for n, a in ARTS.items()}  # test rows are selected only in the final step
FINGERPRINTS = {n: a.fingerprint for n, a in ARTS.items()}
print({n: (X_TRAIN[n].shape, a.fingerprint.get("revision", a.fingerprint.get("sha256", "")[:12])) for n, a in ARTS.items()})

In [ ]:
# Checkpoint identity: code, config, split, embedding artifacts and library versions.
CODE = {n: f"notebooks/{n}" for n in ("filtering.py", "filter.toml", "textprep.py", "embeddings.py", "splits.py",
                                      "taxonomy.py", "taxonomy.toml", "checkpoints.py")}
DATA = {"manifest": SPLIT_MANIFEST_CSV, "train": SPLIT_TRAIN_CSV, "test": SPLIT_TEST_CSV}
DATA.update({f"embedding-{n}": EMBEDDINGS_DIR / f"{embeddings.artifact_name('corpus', n)}.json" for n in EMBEDDER_NAMES})
IDENTITY = ck.make_identity(
    code_paths=CODE, data_paths=DATA, modules=["numpy", "pandas", "scikit-learn", "xgboost", "hyperopt", "scipy"],
    config={"filter": CFG, "seed": RANDOM_STATE, "manifest_sha256": SPLIT_MANIFEST_SHA256, "device": DEVICE,
            "textprep_version": TEXTPREP_VERSION, "fingerprints": FINGERPRINTS,
            "npy_sha256": {n: a.meta["npy_sha256"] for n, a in ARTS.items()}})
STORE = ck.Store.from_env(IDENTITY)
BUDGET = ck.Budget.from_env()
STOPPED = False
EXPECTED = [fl.candidate_key(c.name) for c in CANDIDATES]
print("identity", IDENTITY["digest"][:12], "| store", STORE.out_dir)

## Selection: grouped 5-fold CV of all candidates (train only)

In [ ]:
RECORDS = fl.run_candidates(CANDIDATES, CFG, lambda e, ids: ARTS[e].select(ids), S.y_train, S.folds, S.train_ids, S.test_ids,
                            seed=RANDOM_STATE, device=DEVICE, store=STORE, budget=BUDGET)
if len(RECORDS) < len(CANDIDATES):
    STOPPED = True
    done = [fl.candidate_key(n) for n in RECORDS]
    ck.write_status(STORE.out_dir, "INCOMPLETE", done=done, missing=[k for k in EXPECTED if k not in done],
                    identity_digest=STORE.digest)
    print(f"INCOMPLETE: {len(RECORDS)}/{len(CANDIDATES)} candidates scored within the budget; rerun to resume.")

In [ ]:
if not STOPPED:
    ck.verify_complete(STORE, EXPECTED)
    ORDER = fl.rank_candidates(RECORDS, CFG["selection"]["metric"])
    PERF = fl.performance_table(RECORDS, ORDER)
    BEST = ORDER[0]  # highest mean CV balanced accuracy; ties: lower std, then name
    BEST_CAND = next(c for c in CANDIDATES if c.name == BEST)
    print(f"selected: {BEST} (CV balanced accuracy {PERF.loc[BEST, 'CV Balanced Accuracy']:.2f} +- {PERF.loc[BEST, 'CV Std']:.2f})")
    print("CV scores are SELECTION DIAGNOSTICS (best of the candidates, optimistically biased); each is the mean over the 5 folds of the fold metric.")
    display(PERF)

## Tuning: hyperopt TPE on the selected family (grouped CV on train only)

In [ ]:
if not STOPPED:
    TUNING = fl.run_tuning(BEST, BEST_CAND.family, CFG, X_TRAIN[BEST_CAND.embedder], S.y_train, S.folds, S.train_ids,
                           S.test_ids, seed=RANDOM_STATE, device=DEVICE, store=STORE, budget=BUDGET, max_evals=MAX_EVALS)
    if not TUNING["complete"]:
        STOPPED = True
        tdone = [fl.trial_key(BEST, i) for i in range(TUNING["n_done"])]
        ck.write_status(STORE.out_dir, "INCOMPLETE", done=EXPECTED + tdone,
                        missing=[fl.trial_key(BEST, i) for i in range(TUNING["n_done"], MAX_EVALS)], identity_digest=STORE.digest)
        print(f"INCOMPLETE: {TUNING['n_done']}/{MAX_EVALS} tuning trials within the budget; rerun to resume.")
    else:
        TKEYS = [fl.trial_key(BEST, i) for i in range(MAX_EVALS)]
        ck.verify_complete(STORE, TKEYS)
        print("best params:", TUNING["best_params"], "| CV balanced accuracy", round(TUNING["best_cv"]["balanced_accuracy_mean"] * 100, 2))

## Refit on all train rows and one test report

In [ ]:
if not STOPPED:
    fam, emb = BEST_CAND.family, BEST_CAND.embedder
    X_test = ARTS[emb].select(S.test_ids)  # test features first selected here, after every choice is made
    TEST_GROUPS = manifest.set_index(manifest["row_id"].astype(str)).loc[S.test_ids, "group"].tolist()  # resampling units
    BOOT_KW = dict(B=int(BOOT["B"]), seed=int(BOOT["seed"]))
    TUNED = fl.make_pipeline(CFG, fam, RANDOM_STATE, DEVICE, TUNING["best_params"]).fit(X_TRAIN[emb], S.y_train)
    TEST_TUNED = fl.evaluate(TUNED, X_test, S.y_test, TEST_GROUPS, **BOOT_KW)
    # Pre-planned descriptive comparison (untuned selected candidate); never used for any choice.
    REFERENCE = fl.make_pipeline(CFG, fam, RANDOM_STATE, DEVICE).fit(X_TRAIN[emb], S.y_train)
    TEST_UNTUNED = fl.evaluate(REFERENCE, X_test, S.y_test, TEST_GROUPS, **BOOT_KW)

    def show(d):
        out = {k: round(v * 100, 2) for k, v in d.items() if k in fl.METRICS or k == "prevalence"}
        out["confusion_matrix"], out["n"] = d["confusion_matrix"], d["n"]
        if "bootstrap" in d:
            b = d["bootstrap"]
            out[f"95% CI ({b['unit']} bootstrap, B={b['B']})"] = {k: [round(x * 100, 2) for x in v] for k, v in b["ci"].items()}
        return out
    print("TEST, tuned (deployed):", show(TEST_TUNED))
    print("TEST, untuned selected candidate (pre-planned descriptive comparison, not used for choice):", show(TEST_UNTUNED))

## Save the deployed filter and the per-candidate CV table

In [ ]:
if not STOPPED:
    def lib_versions():
        import importlib.metadata as md
        return {p: md.version(p) for p in ("numpy", "pandas", "scikit-learn", "xgboost", "hyperopt", "joblib", "scipy")}

    # Cotov precheck, labels and artifact come first: nothing is written under a final name before the
    # Cotov evaluation (next section) succeeds or is explicitly blocked.
    problem = None
    if not (COTOV_CSV and os.path.exists(COTOV_CSV)):
        problem = "Cotov dataset unavailable (COTOV_CSV, used for the Related labels)"
    elif not (EMBEDDINGS_DIR / f"{embeddings.artifact_name('cotov', BEST_CAND.embedder)}.json").exists():
        problem = f"no cotov embedding artifact for {BEST_CAND.embedder} in EMBEDDINGS_DIR"
    # Production fails closed here (no COMPLETE status) unless FILTER_ALLOW_NO_COTOV=1; smoke may skip.
    BLOCKED = fl.require_cotov(SMOKE, ALLOW_NO_COTOV, problem)
    if BLOCKED:
        print("BLOCKED:", BLOCKED)
    else:
        c_ids, c_texts, c_y = fl.cotov_rows(pd.read_csv(COTOV_CSV))
        COTOV_ART = embeddings.load_artifact(
            EMBEDDINGS_DIR, embeddings.artifact_name("cotov", BEST_CAND.embedder), expected_row_ids=c_ids,
            expected_fingerprint=FINGERPRINTS[BEST_CAND.embedder], expected_texts_sha256=embeddings.texts_sha256(c_ids, c_texts),
            allow_unresolved=ALLOW_UNRESOLVED)
        COTOV_X = COTOV_ART.select(c_ids)

    Path(OUT_PICKLE).parent.mkdir(parents=True, exist_ok=True)
    TMP_PICKLE, TMP_CSV = fl.staged(OUT_PICKLE), fl.staged(OUT_CSV)
    joblib.dump(TUNED, TMP_PICKLE, compress=9)
    SIDECAR = {
        "non_production": SMOKE,
        "model": BEST, "family": BEST_CAND.family,
        "embedder": {"name": BEST_CAND.embedder, "fingerprint": FINGERPRINTS[BEST_CAND.embedder]},
        "textprep_version": TEXTPREP_VERSION, "manifest_sha256": SPLIT_MANIFEST_SHA256,
        "random_state": RANDOM_STATE, "params": TUNING["best_params"], "max_evals": MAX_EVALS,
        "cv_note": "CV scores are selection diagnostics (selection-biased); each is the mean (std, ddof=1) over the 5 grouped folds of the fold metric.",
        "cv_untuned_selection_diagnostic": {"mean": RECORDS[BEST]["mean"], "std": RECORDS[BEST]["std"]},
        "cv_tuned_selection_diagnostic": {"mean": TUNING["best_cv"]["mean"], "balanced_accuracy_std": TUNING["best_cv"]["std"]},
        "test_tuned": TEST_TUNED,
        "test_untuned_reference": {"label": "pre-planned descriptive comparison, never used for choice", **TEST_UNTUNED},
        "bootstrap": {**BOOT, "test_unit": "manifest group", "cotov_unit": "row (no groups)"},
        "pickle_sha256": ck.file_sha256(TMP_PICKLE), "identity_digest": IDENTITY["digest"],
        "library_versions": lib_versions(), "python": platform.python_version(),
    }
    PERF.to_csv(TMP_CSV, header=True, sep="&")
    print("staged", TMP_PICKLE, "and", TMP_CSV, "(renamed to the final names after the Cotov evaluation)")
    DEPLOYED = joblib.load(TMP_PICKLE)
    assert np.array_equal(DEPLOYED.predict(X_test), TUNED.predict(X_test))


# Performance over Cotov's Dataset
External evaluation of the deployed (tuned, refit) filter; nothing here is used for any fit or choice.

In [ ]:
if not STOPPED:
    if BLOCKED:
        SIDECAR["cotov"], SIDECAR["cotov_blocked_reason"] = "blocked", BLOCKED
    else:
        COTOV = fl.evaluate(DEPLOYED, COTOV_X, c_y, None, **BOOT_KW)  # no groups: rows are resampled
        print(f"Cotov ({COTOV['n']} English rows, prevalence of relevant {c_y.mean():.3f}):", show(COTOV))
        SIDECAR["cotov"] = COTOV
    # Only now do the pickle, CSV and sidecar appear under their final names (sidecar last, with the status).
    fl.publish_filter(OUT_PICKLE, OUT_CSV, SIDECAR, BLOCKED)
    print("saved", OUT_PICKLE, "+ .json sidecar,", OUT_CSV)


In [ ]:
if not STOPPED:
    ck.write_status(STORE.out_dir, "COMPLETE", done=EXPECTED + TKEYS, identity_digest=STORE.digest)
    print("COMPLETE", "(NON-PRODUCTION smoke)" if SMOKE else "")